# RAG Assignment: Baseline Retrieval vs. Cross-Encoder Re-Ranking

**Document:** The Digital Personal Data Protection Act, 2023 (India) - a 21-page legal text with many
exceptions ("shall not apply"), prohibitions ("shall not") and a penalty Schedule located far from the rules it refers to.

Two pipelines are run over the **same chunked corpus**:

| | Pipeline 1 (Baseline) | Pipeline 2 (Re-ranking) |
|---|---|---|
| Stage 1 | Bi-Encoder + FAISS, Top 10 by cosine similarity | same Top 10 |
| Stage 2 | - | Cross-Encoder scores all 10 (query, chunk) pairs, re-sort |
| LLM context | Top 3 of stage 1 -> **Answer 1** | Top 3 after re-ranking -> **Answer 2** |

Notebook outline:
1. Setup and configuration
2. PDF loading and cleaning
3. Chunking
4. Pipeline 1 - Bi-Encoder index and Top-10 retrieval
5. Pipeline 2 - Cross-Encoder re-ranking
6. LLM answer generation (identical prompt for both pipelines)
7. Failure-mode queries (keyword trap, exception, negation, buried chunk)
8. Rank-shift analysis and Answer 1 vs Answer 2
9. Cases where re-ranking does not help
10. Latency profiling
11. Scaling experiment (Cross-Encoder over the whole corpus)

## 1. Setup and configuration

In [1]:
# Install once if needed:
# %pip install -q sentence-transformers faiss-cpu pymupdf transformers torch pandas matplotlib google-genai

import os
# FAISS and PyTorch each ship their own OpenMP runtime on macOS; with more than one thread the
# kernel can crash, so OpenMP is limited to one thread. This must be set before the imports.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re, time, json, hashlib, statistics, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf                      # PDF text extraction
import faiss                        # vector store
import torch
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, CrossEncoder

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

In [2]:
# ---------------- configuration ----------------
PDF_PATH = Path("data/DPDP_Act_2023.pdf")
PDF_URL  = "https://www.meity.gov.in/static/uploads/2024/06/2bf1f0e9f04e6fb4f8fef35e82c42aa5.pdf"

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"     # dense embedding model (384-d)
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"       # re-ranker
LOCAL_LLM_NAME     = "Qwen/Qwen2.5-0.5B-Instruct"                 # open-weights LLM run locally
# used only if GEMINI_API_KEY is set; the free tier allows ~20 requests/day per model, so when one
# model runs out of quota (or is overloaded) the next one in the list is used
GEMINI_MODELS      = ["gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash"]

TOP_K_RETRIEVE = 10    # candidates fetched by the Bi-Encoder
TOP_K_CONTEXT  = 3     # chunks given to the LLM
MAX_CHUNK_WORDS = 130  # keeps a chunk inside the 256-token window of the Bi-Encoder
OVERLAP_PARAS   = 1    # paragraphs repeated between neighbouring chunks

RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
torch.manual_seed(0)

## 2. PDF loading and cleaning

The Gazette PDF is not plain running text, so three things are handled while reading it:

* **Masthead / running headers / signature block** are dropped using their position on the page.
* **Marginal notes.** Each section's title is printed in the page margin (e.g. *"Exemptions."*), outside the
  main text column. Blocks outside the column are separated from the body and the note that sits next to a
  section's first line is attached to that section as its title.
* **The Schedule (penalty table).** Table cells are separate text blocks. Blocks that share the same
  baseline are merged so that each row reads as `breach ... penalty`.

In [3]:
if not PDF_PATH.exists():                      # the PDF is committed in data/, this is only a fallback
    PDF_PATH.parent.mkdir(exist_ok=True)
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    PDF_PATH.write_bytes(urllib.request.urlopen(req).read())

BODY_X0, BODY_X1 = 110, 482      # x-range (points) of the main text column
HEADER_Y = 85                    # running header is above this y on pages 2+
# the margin notes of these two sections are missing from the PDF text layer, so they are typed in
MISSING_TITLES = {"7": "Certain legitimate uses", "25": "Members and officers to be public servants"}

def clean(text):
    return re.sub(r"\s+", " ", text).strip()

def extract_paragraphs(pdf_path):
    # Returns a list of paragraphs, each tagged with its page and section number / title.
    doc = pymupdf.open(pdf_path)
    paras = []
    sec, title = "Preamble", "Long title"
    for pno, page in enumerate(doc, start=1):
        blocks = [b for b in page.get_text("blocks") if b[6] == 0]          # text blocks only
        # marginal notes (section titles); "24 of 1997." style citations are ignored
        notes = [(b[1], clean(b[4])) for b in blocks
                 if (b[0] >= BODY_X1 or b[2] <= BODY_X0)
                 and not re.match(r"^\d+ of \d{4}\.$", clean(b[4]))]
        body = [b for b in blocks if b[0] >= BODY_X0 and b[2] <= BODY_X1 + 2]
        body = [b for b in body if b[1] >= (510 if pno == 1 else HEADER_Y)]  # masthead / header
        if pno == len(doc):
            body = [b for b in body if b[1] < 550]                           # signature and imprint
        rows = {}                                                            # group blocks by baseline
        for b in sorted(body, key=lambda b: (round(b[1] / 4), b[0])):
            rows.setdefault(round(b[1] / 4), []).append(b)
        for key in sorted(rows):
            text = clean(" ".join(b[4] for b in rows[key]))
            y = rows[key][0][1]
            if not text or re.fullmatch(r"[—–_\-\s]+", text):
                continue
            if re.fullmatch(r"CHAPTER [IVX]+", text) or (text.isupper() and len(text) < 80 and not text.startswith("THE")):
                continue                                                     # chapter number / chapter name lines
            if text.startswith("THE SCHEDULE"):
                sec, title = "Schedule", "Penalties for breach"
            m = re.match(r"^(\d{1,2})\.\s", text)
            # a new section starts only when the number is the next one in sequence
            if m and sec != "Schedule" and int(m.group(1)) == (int(sec) + 1 if sec.isdigit() else 1):
                near = [n for ny, n in notes if abs(ny - y) < 8]
                sec = m.group(1)
                title = near[0].rstrip(".") if near else MISSING_TITLES.get(sec, "")
            paras.append({"page": pno, "text": text, "sec": sec, "title": title})
    return paras

paragraphs = extract_paragraphs(PDF_PATH)
print(f"pages: {len(pymupdf.open(PDF_PATH))} | paragraphs: {len(paragraphs)} | "
      f"words: {sum(len(p['text'].split()) for p in paragraphs)} | "
      f"sections found: {len({p['sec'] for p in paragraphs})}")
pd.DataFrame(paragraphs).drop_duplicates("sec")[["sec", "title", "page"]].head(12)

pages: 21 | paragraphs: 355 | words: 10040 | sections found: 46


,sec,title,page
0,Preamble,Long title,1
5,1,Short title and commencement,1
7,2,Definitions,2
49,3,Application of Act,3
61,4,Grounds for processing personal data,4
65,5,Notice,4
81,6,Consent,5
98,7,Certain legitimate uses,6
117,8,General obligations of Data Fiduciary,7
140,9,Processing of personal data of children,8


## 3. Chunking

Section-aware chunking: consecutive paragraphs of the **same section** are packed together until about
130 words, with one paragraph of overlap so a sub-clause is not cut off from its lead-in. A chunk never
crosses a section boundary. Every chunk is prefixed with `[Section N - title]` so that a sub-clause still
carries the topic of its section. Both pipelines use exactly this list of chunks.

In [4]:
def build_chunks(paras, max_words=MAX_CHUNK_WORDS, overlap=OVERLAP_PARAS):
    chunks, cur = [], []

    def flush():
        if not cur:
            return
        first = cur[0]
        label = (f"Section {first['sec']}" if first["sec"].isdigit() else first["sec"]) + f" - {first['title']}"
        chunks.append({"id": len(chunks), "section": first["sec"], "title": first["title"],
                       "pages": sorted({p["page"] for p in cur}),
                       "text": f"[{label}] " + " ".join(p["text"] for p in cur)})

    for p in paras:
        words_now = sum(len(x["text"].split()) for x in cur)
        new_section = bool(cur) and p["sec"] != cur[0]["sec"]
        if cur and (new_section or words_now + len(p["text"].split()) > max_words):
            flush()
            cur = [] if new_section else cur[-overlap:]
            if cur and len(cur[0]["text"].split()) > max_words // 2:   # do not repeat a very long paragraph
                cur = []
        cur.append(p)
    flush()
    return chunks

chunks = build_chunks(paragraphs)
chunk_texts = [c["text"] for c in chunks]
lengths = [len(t.split()) for t in chunk_texts]
print(f"{len(chunks)} chunks | words per chunk: min {min(lengths)}, median {statistics.median(lengths):.0f}, max {max(lengths)}")
for i in (42, len(chunks) - 2):
    print(f"\n--- chunk {i} (page {chunks[i]['pages']}) ---\n{chunks[i]['text']}")

120 chunks | words per chunk: min 37, median 112, max 160

--- chunk 42 (page [8]) ---
[Section 9 - Processing of personal data of children] 9. (1) The Data Fiduciary shall, before processing any personal data of a child or a person with disability who has a lawful guardian obtain verifiable consent of the parent of such child or the lawful guardian, as the case may be, in such manner as may be prescribed. Explanation.—For the purpose of this sub-section, the expression “consent of the parent” includes the consent of lawful guardian, wherever applicable. (2) A Data Fiduciary shall not undertake such processing of personal data that is likely to cause any detrimental effect on the well-being of a child. (3) A Data Fiduciary shall not undertake tracking or behavioural monitoring of children or targeted advertising directed at children.

--- chunk 118 (page [21]) ---
[Schedule - Penalties for breach] THE SCHEDULE [See section 33 (1)] Sl. No. Breach of provisions of this Act or rules made 

## 4. Pipeline 1 - Baseline retrieval (Bi-Encoder + FAISS)

Each chunk is embedded **once, independently of any query** with `all-MiniLM-L6-v2`. The vectors are
L2-normalised and stored in a FAISS `IndexFlatIP`; inner product on unit vectors equals cosine similarity.
At query time only the query is embedded and the index returns the Top 10.

In [5]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device="cpu")

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, normalize_embeddings=True, convert_to_numpy=True,
                              batch_size=32, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(chunk_emb.shape[1])     # exact inner-product search
index.add(chunk_emb)
index_build_s = time.perf_counter() - t0
print(f"indexed {index.ntotal} chunks, dim {chunk_emb.shape[1]}, one-off indexing time {index_build_s:.2f}s")

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Pipeline 1: embed the query, return the top-k chunks by cosine similarity (best first).
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return [{"chunk_id": int(i), "bi_rank": r + 1, "bi_score": float(s)}
            for r, (i, s) in enumerate(zip(ids[0], scores[0]))]

indexed 120 chunks, dim 384, one-off indexing time 1.73s


## 5. Pipeline 2 - Cross-Encoder re-ranking

The same Top-10 candidates are passed as `(query, chunk)` pairs through
`cross-encoder/ms-marco-MiniLM-L-6-v2`. Query and chunk go through the transformer **together**, so every
query token can attend to every chunk token. The model outputs one relevance logit per pair; the
candidates are re-sorted by it in descending order.

In [6]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu", max_length=512)

def rerank(query, candidates):
    # Pipeline 2: score all (query, chunk) pairs jointly and re-sort descending by that score.
    scores = cross_encoder.predict([(query, chunks[c["chunk_id"]]["text"]) for c in candidates],
                                   show_progress_bar=False)
    scored = [dict(c, ce_score=float(s)) for c, s in zip(candidates, scores)]
    scored.sort(key=lambda c: -c["ce_score"])
    for r, c in enumerate(scored, start=1):
        c["ce_rank"] = r
    return scored

## 6. LLM answer generation

The same prompt template is used for both pipelines; only the three context chunks differ. The model is
told to answer from the context only, so a missing chunk shows up as a wrong or incomplete answer.
Temperature is 0 / decoding is greedy.

Backend: the Gemini API if `GEMINI_API_KEY` is available (environment variable or a local `.env` file),
otherwise the open-weights `Qwen2.5-0.5B-Instruct` runs locally through Hugging Face `transformers`
(no key needed, but a 0.5B model often misreads legal text).

In [7]:
SYSTEM_PROMPT = ("You answer questions about the Digital Personal Data Protection Act, 2023. "
                 "Use ONLY the context passages. Quote the relevant rule and mention its section. "
                 "If the context does not contain the answer, reply exactly: "
                 "'The provided context does not contain the answer.' Answer in at most 3 sentences.")

def build_prompt(query, context_ids):
    context = "\n\n".join(f"Passage {n}: {chunks[i]['text']}" for n, i in enumerate(context_ids, start=1))
    return f"Context passages:\n{context}\n\nQuestion: {query}\nAnswer:"

# The key is read from the environment or from a local .env file (which is git-ignored, never committed).
if not os.environ.get("GEMINI_API_KEY") and Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if line.strip().startswith("GEMINI_API_KEY="):
            os.environ["GEMINI_API_KEY"] = line.split("=", 1)[1].strip().strip('"\'')
USE_GEMINI = bool(os.environ.get("GEMINI_API_KEY"))

if USE_GEMINI:
    from google import genai                      # pip install google-genai
    from google.genai import types
    gemini = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    LLM_NAME = "Gemini API"
    unavailable = set()                           # models that ran out of daily quota in this session

    def call_llm(query, context_ids):
        # Returns (answer, model used, generation time in ms of the successful request).
        config = types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.0)
        for model in [m for m in GEMINI_MODELS if m not in unavailable]:
            for attempt in range(2):
                try:
                    t0 = time.perf_counter()
                    resp = gemini.models.generate_content(model=model, config=config,
                                                          contents=build_prompt(query, context_ids))
                    return (resp.text or "").strip(), model, (time.perf_counter() - t0) * 1000
                except Exception as err:
                    if "PerDay" in str(err):      # daily quota used up -> next model
                        unavailable.add(model)
                        break
                    time.sleep(15)                # per-minute limit or 503 overload -> wait, retry once
        raise RuntimeError("no Gemini model available (quota exhausted or overloaded)")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    LLM_NAME = LOCAL_LLM_NAME
    llm_device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    llm_tok = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(LOCAL_LLM_NAME, torch_dtype=torch.float32).to(llm_device).eval()

    def call_llm(query, context_ids):
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_prompt(query, context_ids)}]
        inputs = llm_tok.apply_chat_template(messages, add_generation_prompt=True,
                                             return_tensors="pt", return_dict=True).to(llm_device)
        t0 = time.perf_counter()
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=160, do_sample=False, repetition_penalty=1.05)
        text = llm_tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return text, LOCAL_LLM_NAME, (time.perf_counter() - t0) * 1000

# Answers are cached on disk, keyed by the exact prompt, so that re-running the notebook does not spend
# the small free-tier quota again. The model name and its generation time are stored with each answer.
# Delete results/llm_cache.json to regenerate everything.
CACHE_PATH = RESULTS_DIR / "llm_cache.json"
llm_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def generate_answer(query, context_ids):
    # Returns the cache entry: {"answer", "model", "ms", ...}
    key = hashlib.sha256((SYSTEM_PROMPT + build_prompt(query, context_ids)).encode()).hexdigest()[:16]
    if key not in llm_cache:
        answer, model, ms = call_llm(query, context_ids)
        llm_cache[key] = {"model": model, "query": query, "context_ids": list(context_ids),
                          "answer": answer, "ms": round(ms, 1)}
        CACHE_PATH.write_text(json.dumps(llm_cache, indent=2, ensure_ascii=False))
        if USE_GEMINI:
            time.sleep(6)                         # stay under the requests-per-minute limit
    return llm_cache[key]

print("LLM backend:", LLM_NAME, "| cached answers:", len(llm_cache))

LLM backend: Gemini API | cached answers: 18


In [8]:
def run_both(query, generate=True):
    # Runs both pipelines on one query and records the time spent in every stage.
    t0 = time.perf_counter()
    candidates = baseline_retrieve(query)                  # stage 1, shared by both pipelines
    t1 = time.perf_counter()
    reranked = rerank(query, candidates)                   # stage 2, pipeline 2 only
    t2 = time.perf_counter()

    top3_base = [c["chunk_id"] for c in candidates[:TOP_K_CONTEXT]]
    top3_rerank = [c["chunk_id"] for c in reranked[:TOP_K_CONTEXT]]
    out = {"query": query, "candidates": reranked, "top3_baseline": top3_base, "top3_rerank": top3_rerank,
           "t_retrieve_ms": (t1 - t0) * 1000, "t_rerank_ms": (t2 - t1) * 1000}
    if generate:
        a1 = generate_answer(query, top3_base)        # baseline context
        a2 = generate_answer(query, top3_rerank)      # re-ranked context, identical prompt
        out.update(answer_1=a1["answer"], answer_2=a2["answer"], t_llm_1_ms=a1["ms"], t_llm_2_ms=a2["ms"],
                   llm_1=a1["model"], llm_2=a2["model"])
    return out

def rank_table(result, gold_ids=()):
    # One row per candidate, in baseline order, with its rank after re-ranking.
    rows = []
    for c in sorted(result["candidates"], key=lambda c: c["bi_rank"]):
        ch = chunks[c["chunk_id"]]
        rows.append({"baseline rank": c["bi_rank"], "re-ranked rank": c["ce_rank"],
                     "shift": c["bi_rank"] - c["ce_rank"], "chunk": c["chunk_id"],
                     "section": ch["section"], "cosine": round(c["bi_score"], 3),
                     "CE score": round(c["ce_score"], 2),
                     "gold": "<-- answer" if c["chunk_id"] in gold_ids else "",
                     "text": ch["text"][:95] + "..."})
    return pd.DataFrame(rows)

### Sanity check (control query)

A plain question whose wording matches the Act. Both pipelines should already agree here.

In [9]:
control = run_both("Can Board members be sued for actions done in good faith?")
display(rank_table(control).head(5))
print("Baseline top 3 :", control["top3_baseline"], "\nRe-ranked top 3:", control["top3_rerank"])
print("\nAnswer 1:", control["answer_1"], "\n\nAnswer 2:", control["answer_2"])

,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,1,0,100,35,0.551,8.49,,"[Section 35 - Protection of action taken in good faith] 35. No suit, prosecution or other legal..."
1,2,4,-2,73,23,0.515,-7.39,,[Section 23 - Proceedings of Board] 23. (1) The Board shall observe such procedure in regard to...
2,3,2,1,106,39,0.515,-3.19,,[Section 39 - Bar of jurisdiction] 39. No civil court shall have the jurisdiction to entertain ...
3,4,10,-6,77,26,0.462,-10.90,,"[Section 26 - Powers of Chairperson] 26. The Chairperson shall exercise the following powers, n..."
4,5,5,0,94,31,0.444,-8.10,,[Section 31 - Alternate dispute resolution] 31. If the Board is of the opinion that any complai...


Baseline top 3 : [100, 73, 106] 
Re-ranked top 3: [100, 106, 85]

Answer 1: No, Board members cannot be sued for actions done in good faith. According to Section 35, "No suit, prosecution or other legal proceedings shall lie against the Central Government, the Board, its Chairperson and any Member, officer or employee thereof for anything which is done or intended to be done in good faith under the provisions of this Act or the rules made thereunder." 

Answer 2: No, Board members cannot be sued for actions done in good faith. According to Section 35, "No suit, prosecution or other legal proceedings shall lie against the Central Government, the Board, its Chairperson and any Member, officer or employee thereof for anything which is done or intended to be done in good faith under the provisions of this Act or the rules made thereunder."


## 7. Failure-mode queries

Each test query has a **gold passage**, identified by a phrase that appears only in the clause that
answers it (looked up in the Act by hand). A chunk is "gold" if it contains that phrase; because chunks
overlap by one paragraph, there can be two gold chunks.

| ID | Category | Why the baseline is expected to struggle |
|---|---|---|
| F1 | Negation & constraints | "not eligible" - the positive *qualifications* section (s.19) is close in embedding space; the answer is the *disqualifications* list (s.21) |
| F2 | Keyword trap / distractor | "children's data" matches the three s.9 chunks; the amount is in the Schedule, which talks about "breach ... additional obligations" |
| F3 | Exception / corner case | scope question; the "(c) not apply to - personal or domestic purpose" carve-out is in the first s.3 chunk, the baseline prefers the second one |
| F4 | Buried chunk (rank 4-10) | two-part question (appeal + time limit); Board-procedure chunks outrank the appeal section |
| F5 | Exception / corner case | general exemption power in s.17(5) is hidden behind the narrower startup exemption |
| F6 | Keyword trap / distractor | "false information", "frivolous complaint" match the *duties* section; the consequence is in s.28(12) |
| L1 | Limitation: re-ranking hurts | "always required" asks for the exception; see section 9 |
| L2 | Limitation: recall ceiling | gold chunk is not in the Top 10 at all; see section 9 |

In [10]:
TEST_QUERIES = [
    {"id": "F1", "category": "Negation & constraints",
     "query": "Who is not eligible to be a Member of the Board?",
     "gold_phrase": "has been adjudged as an insolvent"},
    {"id": "F2", "category": "Keyword trap / distractor",
     "query": "What is the maximum penalty for not protecting children's data?",
     "gold_phrase": "additional obligations in relation to children under section 9"},
    {"id": "F3", "category": "Exception / corner case",
     "query": "Does the Act cover data processed for personal or domestic purposes?",
     "gold_phrase": "for any personal or domestic purpose"},
    {"id": "F4", "category": "Buried chunk (rank 4-10)",
     "query": "Can the Board's order be challenged, and within how many days?",
     "gold_phrase": "within a period of sixty days"},
    {"id": "F5", "category": "Exception / corner case",
     "query": "Can the Central Government exempt a company from the Act entirely?",
     "gold_phrase": "before expiry of five years"},
    {"id": "F6", "category": "Keyword trap / distractor",
     "query": "What is the punishment for a user who gives false information or files a frivolous complaint?",
     "gold_phrase": "issue a warning or impose costs on the complainant"},
    {"id": "L1", "category": "Limitation: re-ranking hurts",
     "query": "Is parental consent always required before processing a child's personal data?",
     "gold_phrase": "shall not be applicable to processing of personal data of a child"},
    {"id": "L2", "category": "Limitation: recall ceiling",
     "query": "Can I complain to the Board directly without first going to the company?",
     "gold_phrase": "exhaust the opportunity of redressing her grievance"},
]

for t in TEST_QUERIES:      # resolve each gold phrase to chunk ids
    t["gold_ids"] = [c["id"] for c in chunks if t["gold_phrase"] in c["text"]]
    assert t["gold_ids"], f"gold phrase not found for {t['id']}"
pd.DataFrame(TEST_QUERIES)[["id", "category", "query", "gold_ids"]]

,id,category,query,gold_ids
0,F1,Negation & constraints,Who is not eligible to be a Member of the Board?,[70]
1,F2,Keyword trap / distractor,What is the maximum penalty for not protecting children's data?,"[118, 119]"
2,F3,Exception / corner case,Does the Act cover data processed for personal or domestic purposes?,[10]
3,F4,Buried chunk (rank 4-10),"Can the Board's order be challenged, and within how many days?",[88]
4,F5,Exception / corner case,Can the Central Government exempt a company from the Act entirely?,[66]
5,F6,Keyword trap / distractor,What is the punishment for a user who gives false information or files a frivolous complaint?,[87]
6,L1,Limitation: re-ranking hurts,Is parental consent always required before processing a child's personal data?,"[43, 44]"
7,L2,Limitation: recall ceiling,Can I complain to the Board directly without first going to the company?,[53]


In [11]:
results = {}
for t in TEST_QUERIES:
    results[t["id"]] = run_both(t["query"])
    print(t["id"], "done")

F1 done
F2 done


F3 done
F4 done


F5 done
F6 done


L1 done
L2 done


## 8. Rank-shift analysis and Answer 1 vs Answer 2

For every query: the full Top-10 with baseline rank, rank after re-ranking and both scores, followed by
the two LLM answers. `shift > 0` means the chunk moved up.

In [12]:
def best_rank(result, gold_ids, key):
    ranks = [c[key] for c in result["candidates"] if c["chunk_id"] in gold_ids]
    return min(ranks) if ranks else None       # None = gold chunk not among the 10 candidates

def show(t):
    r = results[t["id"]]
    print("=" * 110)
    print(f"{t['id']} [{t['category']}]  {t['query']}")
    print(f"gold phrase: \"{t['gold_phrase']}\"  -> gold chunk(s) {t['gold_ids']}")
    display(rank_table(r, t["gold_ids"]))
    print(f"Context for Answer 1 (baseline top 3) : {r['top3_baseline']}")
    print(f"Context for Answer 2 (re-ranked top 3): {r['top3_rerank']}")
    print(f"\nAnswer 1 (baseline) [{r['llm_1']}]:\n{r['answer_1']}\n\nAnswer 2 (re-ranked) [{r['llm_2']}]:\n{r['answer_2']}\n")

for t in TEST_QUERIES[:6]:
    show(t)

F1 [Negation & constraints]  Who is not eligible to be a Member of the Board?
gold phrase: "has been adjudged as an insolvent"  -> gold chunk(s) [70]


,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,4,-3,68,19,0.549,-2.58,,[Section 19 - Composition and qualifications for appointment of Chairperson and Members] 19. (1...
1,2,8,-6,111,40,0.539,-7.49,,[Section 40 - Power to make rules] (p) the manner of nomination of any other individual by the ...
2,3,7,-4,75,24,0.513,-7.37,,"[Section 24 - Officers and employees of Board] 24. The Board may, with previous approval of the..."
3,4,1,3,70,21,0.512,1.56,<-- answer,[Section 21 - Disqualifications for appointment and continuation as Chairperson and Members of ...
4,5,6,-1,77,26,0.493,-7.01,,"[Section 26 - Powers of Chairperson] 26. The Chairperson shall exercise the following powers, n..."
5,6,3,3,73,23,0.462,-2.32,,[Section 23 - Proceedings of Board] 23. (1) The Board shall observe such procedure in regard to...
6,7,2,5,76,25,0.460,-2.05,,"[Section 25 - Members and officers to be public servants] 25. The Chairperson, Members, officer..."
7,8,5,3,74,23,0.445,-3.78,,"[Section 23 - Proceedings of Board] (c) any irregularity in the procedure of the Board, which d..."
8,9,9,0,96,32,0.436,-7.53,,"[Section 32 - Voluntary undertaking] (3) The Board may, after accepting the voluntary undertaki..."
9,10,10,0,112,40,0.413,-9.67,,[Section 40 - Power to make rules] (u) the terms and conditions of appointment and service of o...


Context for Answer 1 (baseline top 3) : [68, 111, 75]
Context for Answer 2 (re-ranked top 3): [70, 76, 73]

Answer 1 (baseline) [gemini-3.7-flash]:
The provided context does not contain the answer.

Answer 2 (re-ranked) [gemini-3.7-flash]:
According to Section 21(1), a person is disqualified from being appointed or continued as a Member if she "(a) has been adjudged as an insolvent; (b) has been convicted of an offence, which in the opinion of the Central Government, involves moral turpitude; (c) has become physically or mentally incapable of acting as a Member; (d) has acquired such financial or other interest, as is likely to affect prejudicially her functions as a Member; or (e) has so abused her position as to render her continuance in office prejudicial to the public interest."

F2 [Keyword trap / distractor]  What is the maximum penalty for not protecting children's data?
gold phrase: "additional obligations in relation to children under section 9"  -> gold chunk(s) [118, 119]


,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,4,-3,44,9,0.556,-2.32,,[Section 9 - Processing of personal data of children] (4) The provisions of sub-sections (1) an...
1,2,3,-1,42,9,0.525,-2.02,,"[Section 9 - Processing of personal data of children] 9. (1) The Data Fiduciary shall, before p..."
2,3,2,1,43,9,0.521,-1.00,,[Section 9 - Processing of personal data of children] (3) A Data Fiduciary shall not undertake ...
3,4,1,3,118,Schedule,0.504,0.49,<-- answer,[Schedule - Penalties for breach] THE SCHEDULE [See section 33 (1)] Sl. No. Breach of provision...
4,5,5,0,109,40,0.484,-5.55,,[Section 40 - Power to make rules] (h) the manner of publishing the business contact informatio...
5,6,9,-3,55,15,0.483,-9.05,,[Section 15 - Duties of Data Principal] 15. A Data Principal shall perform the following duties...
6,7,7,0,37,8,0.474,-8.35,,"[Section 8 - General obligations of Data Fiduciary] (6) In the event of a personal data breach,..."
7,8,8,0,56,15,0.464,-8.60,,[Section 15 - Duties of Data Principal] (d) to ensure not to register a false or frivolous grie...
8,9,6,3,47,10,0.463,-8.27,,[Section 10 - Additional obligations of Significant Data Fiduciary] (i) periodic Data Protectio...
9,10,10,0,46,10,0.452,-9.83,,[Section 10 - Additional obligations of Significant Data Fiduciary] (iii) be an individual resp...


Context for Answer 1 (baseline top 3) : [44, 42, 43]
Context for Answer 2 (re-ranked top 3): [118, 43, 42]

Answer 1 (baseline) [gemini-3.7-flash]:
The provided context does not contain the answer.

Answer 2 (re-ranked) [gemini-3.7-flash]:
According to The Schedule [See section 33 (1)], for a "Breach in observance of additional obligations in relation to children under section 9", the penalty "[m]ay extend to two hundred crore rupees."

F3 [Exception / corner case]  Does the Act cover data processed for personal or domestic purposes?
gold phrase: "for any personal or domestic purpose"  -> gold chunk(s) [10]


...

Context for Answer 1 (baseline top 3) : [87, 73, 86]
Context for Answer 2 (re-ranked top 3): [88, 90, 87]

Answer 1 (baseline) [gemini-3.7-flash]:
The provided context does not contain the answer.

Answer 2 (re-ranked) [gemini-3.7-flash]:
Yes, the Board's order can be challenged, as Section 29(1) provides that "[a]ny person aggrieved by an order or direction made by the Board under this Act may prefer an appeal before the Appellate Tribunal." According to Section 29(2), this appeal must be filed "within a period of sixty days from the date of receipt of the order or direction appealed against".

F5 [Exception / corner case]  Can the Central Government exempt a company from the Act entirely?
gold phrase: "before expiry of five years"  -> gold chunk(s) [66]


,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,3,-2,100,35,0.491,-0.31,,"[Section 35 - Protection of action taken in good faith] 35. No suit, prosecution or other legal..."
1,2,1,1,64,17,0.461,0.90,,"[Section 17 - Exemptions] (3) The Central Government may, having regard to the volume and natur..."
2,3,5,-2,65,17,0.439,-1.80,,"[Section 17 - Exemptions] Explanation.—For the purposes of this sub-section, the term “startup”..."
3,4,6,-2,114,42,0.432,-1.96,,"[Section 42 - Power to amend Schedule] 42. (1) The Central Government may, by notification, ame..."
4,5,8,-3,60,17,0.429,-5.46,,[Section 17 - Exemptions] (d) personal data of Data Principals not within the territory of Indi...
5,6,2,4,66,17,0.422,0.88,<-- answer,[Section 17 - Exemptions] (4) In respect of processing by the State or any instrumentality of t...
6,7,10,-3,106,39,0.419,-9.29,,[Section 39 - Bar of jurisdiction] 39. No civil court shall have the jurisdiction to entertain ...
7,8,7,1,101,36,0.382,-2.27,,"[Section 36 - Power to call for information] 36. The Central Government may, for the purposes o..."
8,9,4,5,63,17,0.366,-1.33,,"[Section 17 - Exemptions] (b) necessary for research, archiving or statistical purposes if the ..."
9,10,9,1,102,37,0.364,-5.80,,[Section 37 - Power of Central Government to issue directions] 37. (1) The Central Government o...


Context for Answer 1 (baseline top 3) : [100, 64, 65]
Context for Answer 2 (re-ranked top 3): [64, 66, 100]

Answer 1 (baseline) [gemini-3.6-flash]:
The provided context does not contain the answer.

Answer 2 (re-ranked) [gemini-3.6-flash]:
Yes, the Central Government can exempt a Data Fiduciary from provisions of the Act under Section 17(5). Section 17(5) states, "The Central Government may, before expiry of five years from the date of commencement of this Act, by notification, declare that any provision of this Act shall not apply to such Data Fiduciary or classes of Data Fiduciaries for such period as may be specified in the notification."

F6 [Keyword trap / distractor]  What is the punishment for a user who gives false information or files a frivolous complaint?
gold phrase: "issue a warning or impose costs on the complainant"  -> gold chunk(s) [87]


,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,6,-5,97,33,0.438,-8.61,,[Section 33 - Penalties] 33. (1) If the Board determines on conclusion of an inquiry that breac...
1,2,1,1,56,15,0.433,1.41,,[Section 15 - Duties of Data Principal] (d) to ensure not to register a false or frivolous grie...
2,3,4,-1,79,27,0.430,-5.25,,[Section 27 - Powers and functions of Board] (a) on receipt of an intimation of personal data b...
3,4,3,1,80,27,0.407,-5.13,,[Section 27 - Powers and functions of Board] (c) on a complaint made by a Data Principal in res...
4,5,5,0,18,6,0.391,-8.53,,[Section 6 - Consent] (2) Any part of consent referred in sub-section (1) which constitutes an ...
5,6,8,-2,15,5,0.388,-9.54,,"[Section 5 - Notice] (a) the Data Fiduciary shall, as soon as it is reasonably practicable, giv..."
6,7,7,0,98,33,0.373,-9.39,,"[Section 33 - Penalties] (d) whether the person, as a result of the breach, has realised a gain..."
7,8,2,6,87,28,0.369,-1.31,<-- answer,"[Section 28 - Procedure to be followed by Board] (10) During the course of the inquiry, if the ..."
8,9,9,0,37,8,0.367,-10.27,,"[Section 8 - General obligations of Data Fiduciary] (6) In the event of a personal data breach,..."
9,10,10,0,40,8,0.367,-10.94,,[Section 8 - General obligations of Data Fiduciary] (b) exercise any of her rights in relation ...


Context for Answer 1 (baseline top 3) : [97, 56, 79]
Context for Answer 2 (re-ranked top 3): [56, 87, 80]

Answer 1 (baseline) [gemini-3.6-flash]:
The provided context does not contain the answer.

Answer 2 (re-ranked) [gemini-3.6-flash]:
According to Section 28(12), if the Board is of the opinion that a complaint is false or frivolous, "it may issue a warning or impose costs on the complainant." Additionally, Section 15 sets the duties of a Data Principal "to ensure not to register a false or frivolous grievance or complaint" and "to furnish only such information as is verifiably authentic".



In [13]:
summary = []
for t in TEST_QUERIES:
    r = results[t["id"]]
    b, c = best_rank(r, t["gold_ids"], "bi_rank"), best_rank(r, t["gold_ids"], "ce_rank")
    summary.append({"id": t["id"], "category": t["category"], "query": t["query"],
                    "gold chunk rank (baseline)": b if b else ">10",
                    "gold chunk rank (re-ranked)": c if c else ">10",
                    "gold in baseline top 3": bool(b and b <= 3),
                    "gold in re-ranked top 3": bool(c and c <= 3)})
summary_df = pd.DataFrame(summary)
summary_df.to_csv(RESULTS_DIR / "rank_shift_summary.csv", index=False)
summary_df

,id,category,query,gold chunk rank (baseline),gold chunk rank (re-ranked),gold in baseline top 3,gold in re-ranked top 3
0,F1,Negation & constraints,Who is not eligible to be a Member of the Board?,4,1,False,True
1,F2,Keyword trap / distractor,What is the maximum penalty for not protecting children's data?,4,1,False,True
2,F3,Exception / corner case,Does the Act cover data processed for personal or domestic purposes?,4,1,False,True
3,F4,Buried chunk (rank 4-10),"Can the Board's order be challenged, and within how many days?",4,1,False,True
4,F5,Exception / corner case,Can the Central Government exempt a company from the Act entirely?,6,2,False,True
5,F6,Keyword trap / distractor,What is the punishment for a user who gives false information or files a frivolous complaint?,8,2,False,True
6,L1,Limitation: re-ranking hurts,Is parental consent always required before processing a child's personal data?,2,5,True,False
7,L2,Limitation: recall ceiling,Can I complain to the Board directly without first going to the company?,>10,>10,False,False


In [14]:
# Gold passage text for each query, to judge the answers against
for t in TEST_QUERIES:
    cid = t["gold_ids"][0]
    print(f"{t['id']}  gold chunk {cid}: {chunks[cid]['text']}\n")

F1  gold chunk 70: [Section 21 - Disqualifications for appointment and continuation as Chairperson and Members of Board] 21. (1) A person shall be disqualified for being appointed and continued as the Chairperson or a Member, if she— (a) has been adjudged as an insolvent; (b) has been convicted of an offence, which in the opinion of the Central Government, involves moral turpitude; (c) has become physically or mentally incapable of acting as a Member; (d) has acquired such financial or other interest, as is likely to affect prejudicially her functions as a Member; or (e) has so abused her position as to render her continuance in office prejudicial to the public interest. (2) The Chairperson or Member shall not be removed from her office by the Central Government unless she has been given an opportunity of being heard in the matter.

F2  gold chunk 118: [Schedule - Penalties for breach] THE SCHEDULE [See section 33 (1)] Sl. No. Breach of provisions of this Act or rules made thereunder P

## 9. Cases where re-ranking does not help

Two queries are kept on purpose because they show the limits of the second stage:

* **L1** - the exception chunks are already in the baseline Top 3 and the Cross-Encoder pushes them *down*
  in favour of generic "consent" chunks.
* **L2** - the gold chunk is not among the 10 candidates, so no re-ordering can recover it. The re-ranker
  is bounded by the recall of the first stage.

In [15]:
for t in TEST_QUERIES[6:]:
    show(t)

L1 [Limitation: re-ranking hurts]  Is parental consent always required before processing a child's personal data?
gold phrase: "shall not be applicable to processing of personal data of a child"  -> gold chunk(s) [43, 44]


,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,1,0,42,9,0.707,7.55,,"[Section 9 - Processing of personal data of children] 9. (1) The Data Fiduciary shall, before p..."
1,2,5,-3,44,9,0.627,1.81,<-- answer,[Section 9 - Processing of personal data of children] (4) The provisions of sub-sections (1) an...
2,3,7,-4,43,9,0.597,1.32,<-- answer,[Section 9 - Processing of personal data of children] (3) A Data Fiduciary shall not undertake ...
3,4,10,-6,3,2,0.544,-1.02,,[Section 2 - Definitions] (f) “child” means an individual who has not completed the age of eigh...
4,5,2,3,20,6,0.535,3.71,,[Section 6 - Consent] (4) Where consent given by the Data Principal is the basis of processing ...
5,6,6,0,12,4,0.521,1.39,,[Section 4 - Grounds for processing personal data] 4. (1) A person may process the personal dat...
6,7,3,4,22,6,0.520,3.64,,[Section 6 - Consent] (6) If a Data Principal withdraws her consent to the processing of person...
7,8,9,-1,51,12,0.518,0.49,,[Section 12 - Right to correction and erasure of personal data] 12. (1) A Data Principal shall ...
8,9,8,1,48,11,0.509,0.62,,[Section 11 - Right to access information about personal data] 11. (1) The Data Principal shall...
9,10,4,6,17,6,0.507,2.48,,"[Section 6 - Consent] 6. (1) The consent given by the Data Principal shall be free, specific, i..."


Context for Answer 1 (baseline top 3) : [42, 44, 43]
Context for Answer 2 (re-ranked top 3): [42, 20, 22]

Answer 1 (baseline) [gemini-3.6-flash]:
No, parental consent is not always required before processing a child's personal data. Under Section 9(4), "The provisions of sub-sections (1) and (3) shall not be applicable to processing of personal data of a child by such classes of Data Fiduciaries or for such purposes, and subject to such conditions, as may be prescribed." Additionally, Section 9(5) allows the Central Government to exempt certain Data Fiduciaries from sub-section (1) if processing "is done in a manner that is verifiably safe".

Answer 2 (re-ranked) [gemini-3.6-flash]:
Section 9(1) states that "[t]he Data Fiduciary shall, before processing any personal data of a child or a person with disability who has a lawful guardian obtain verifiable consent of the parent of such child or the lawful guardian, as the case may be, in such manner as may be prescribed." Therefore, verif

,baseline rank,re-ranked rank,shift,chunk,section,cosine,CE score,gold,text
0,1,2,-1,87,28,0.590,-4.96,,"[Section 28 - Procedure to be followed by Board] (10) During the course of the inquiry, if the ..."
1,2,3,-1,83,28,0.509,-6.48,,[Section 28 - Procedure to be followed by Board] 28. (1) The Board shall function as an indepen...
2,3,4,-1,73,23,0.417,-8.14,,[Section 23 - Proceedings of Board] 23. (1) The Board shall observe such procedure in regard to...
3,4,1,3,94,31,0.416,-3.54,,[Section 31 - Alternate dispute resolution] 31. If the Board is of the opinion that any complai...
4,5,7,-2,84,28,0.396,-10.50,,[Section 28 - Procedure to be followed by Board] (4) In case the Board determines that there ar...
5,6,6,0,82,27,0.395,-10.22,,"[Section 27 - Powers and functions of Board] (2) The Board may, for the effective discharge of ..."
6,7,10,-3,74,23,0.377,-10.66,,"[Section 23 - Proceedings of Board] (c) any irregularity in the procedure of the Board, which d..."
7,8,8,0,67,18,0.374,-10.58,,[Section 18 - Establishment of Board] 18. (1) With effect from such date as the Central Governm...
8,9,5,4,86,28,0.372,-8.84,,[Section 28 - Procedure to be followed by Board] (d) such other matters as may be prescribed. (...
9,10,9,1,81,27,0.349,-10.59,,[Section 27 - Powers and functions of Board] (e) on a reference made by the Central Government ...


Context for Answer 1 (baseline top 3) : [87, 83, 73]
Context for Answer 2 (re-ranked top 3): [94, 87, 83]

Answer 1 (baseline) [gemini-3.5-flash]:
The provided context does not contain the answer.

Answer 2 (re-ranked) [gemini-3.6-flash]:
The provided context does not contain the answer.



## 10. Latency profiling

Retrieval stages are timed separately from generation, after a warm-up call, over 30 repetitions of each
of the 8 test queries (240 measurements per stage). Everything runs on CPU so the two stages are
comparable.

* **Baseline retrieval** = query embedding + FAISS Top-10 search
* **Re-ranking pipeline** = baseline retrieval + Cross-Encoder over 10 pairs + sort

In [16]:
N_REPEATS = 30
queries = [t["query"] for t in TEST_QUERIES]
for q in queries[:2]:                       # warm-up
    rerank(q, baseline_retrieve(q))

t_base, t_ce = [], []
for _ in range(N_REPEATS):
    for q in queries:
        a = time.perf_counter()
        cands = baseline_retrieve(q)
        b = time.perf_counter()
        rerank(q, cands)
        c = time.perf_counter()
        t_base.append((b - a) * 1000)
        t_ce.append((c - b) * 1000)
t_base, t_ce = np.array(t_base), np.array(t_ce)
t_total = t_base + t_ce

def stats(x):
    return {"mean (ms)": x.mean(), "median (ms)": np.median(x), "std (ms)": x.std(), "p95 (ms)": np.percentile(x, 95)}

latency_df = pd.DataFrame({
    "Pipeline 1: baseline retrieval (embed query + FAISS top-10)": stats(t_base),
    "Extra cost of Cross-Encoder (10 pairs)": stats(t_ce),
    "Pipeline 2: retrieval + re-ranking": stats(t_total),
}).T.round(2)
latency_df["x baseline"] = (latency_df["mean (ms)"] / t_base.mean()).round(2)
latency_df.to_csv(RESULTS_DIR / "latency_retrieval.csv")
print(f"{len(t_base)} measurements per stage | device: CPU | one-off indexing of {len(chunks)} chunks: {index_build_s:.2f}s")
latency_df

240 measurements per stage | device: CPU | one-off indexing of 120 chunks: 1.73s


,mean (ms),median (ms),std (ms),p95 (ms),x baseline
Pipeline 1: baseline retrieval (embed query + FAISS top-10),5.27,5.19,0.27,5.89,1.00
Extra cost of Cross-Encoder (10 pairs),130.54,132.21,8.90,142.58,24.78
Pipeline 2: retrieval + re-ranking,135.81,137.45,9.00,147.74,25.78


In [17]:
# End-to-end time including generation (single run per query, from section 7).
# Retrieval / re-rank here are single un-warmed runs; the table above is the proper measurement for them.
e2e = pd.DataFrame([{"id": k,
                     "retrieve (ms)": r["t_retrieve_ms"], "re-rank (ms)": r["t_rerank_ms"],
                     "LLM answer 1 (ms)": r["t_llm_1_ms"], "LLM answer 2 (ms)": r["t_llm_2_ms"],
                     "Pipeline 1 total (ms)": r["t_retrieve_ms"] + r["t_llm_1_ms"],
                     "Pipeline 2 total (ms)": r["t_retrieve_ms"] + r["t_rerank_ms"] + r["t_llm_2_ms"]}
                    for k, r in results.items()]).set_index("id").round(1)
e2e.loc["median"] = e2e.median().round(1)
e2e.to_csv(RESULTS_DIR / "latency_end_to_end.csv")
print("LLM:", LLM_NAME, "- generation time is the time of the API request (network + server queue), taken from the cache")
print("NaN = the request had to be retried (model overloaded / rate limited), so no clean time was recorded")
e2e

LLM: Gemini API - generation time is the time of the API request (network + server queue), taken from the cache
NaN = the request had to be retried (model overloaded / rate limited), so no clean time was recorded


,retrieve (ms),re-rank (ms),LLM answer 1 (ms),LLM answer 2 (ms),Pipeline 1 total (ms),Pipeline 2 total (ms)
id,,,,,,
F1,57.6,178.0,8103.3,8580.5,8160.9,8816.1
F2,5.9,156.2,13343.0,NaN,13348.9,NaN
F3,5.9,146.2,NaN,NaN,NaN,NaN
F4,6.6,160.2,NaN,NaN,NaN,NaN
F5,7.4,135.9,5489.9,7154.9,5497.3,7298.2
F6,6.6,136.2,8800.5,6673.4,8807.1,6816.1
L1,5.7,136.1,6814.6,5820.3,6820.3,5962.1
L2,5.4,141.1,11303.8,3612.9,11309.2,3759.3
median,6.2,143.6,8451.9,6673.4,8484.0,6816.1


## 11. Scaling experiment - Cross-Encoder over the whole corpus

To put a number on the scaling question, the Cross-Encoder is run over **every** chunk for a query (no
first stage) and compared with the two-stage pipeline. The Bi-Encoder embeds the corpus once offline;
the Cross-Encoder has to run one forward pass per (query, chunk) pair at query time.

In [18]:
def cross_encode_everything(query, n):
    return cross_encoder.predict([(query, t) for t in chunk_texts[:n]], show_progress_bar=False)

rows = []
for n in (10, 30, 60, len(chunks)):
    times = []
    for q in queries[:4]:
        a = time.perf_counter()
        cross_encode_everything(q, n)
        times.append((time.perf_counter() - a) * 1000)
    rows.append({"chunks scored by Cross-Encoder": n, "mean time per query (ms)": np.mean(times)})
scale_df = pd.DataFrame(rows).round(1)
per_pair_ms = scale_df.iloc[-1, 1] / len(chunks)
scale_df["ms per pair"] = (scale_df.iloc[:, 1] / scale_df.iloc[:, 0]).round(2)
scale_df.to_csv(RESULTS_DIR / "scaling_cross_encoder.csv", index=False)
display(scale_df)

print(f"Two-stage pipeline (this corpus): {t_total.mean():.1f} ms per query")
print(f"Cross-Encoder over all {len(chunks)} chunks: {scale_df.iloc[-1, 1]:.1f} ms per query "
      f"({scale_df.iloc[-1, 1] / t_total.mean():.1f}x slower)")
print(f"\nLinear extrapolation at {per_pair_ms:.2f} ms per pair (one query, this CPU):")
for n in (10_000, 1_000_000, 100_000_000):
    s = n * per_pair_ms / 1000
    print(f"  {n:>11,} chunks -> {s:>12,.0f} s  ({s / 3600:,.1f} h)")

,chunks scored by Cross-Encoder,mean time per query (ms),ms per pair
0,10,146.8,14.68
1,30,455.2,15.17
2,60,950.4,15.84
3,120,1900.8,15.84


Two-stage pipeline (this corpus): 135.8 ms per query
Cross-Encoder over all 120 chunks: 1900.8 ms per query (14.0x slower)

Linear extrapolation at 15.84 ms per pair (one query, this CPU):
       10,000 chunks ->          158 s  (0.0 h)
    1,000,000 chunks ->       15,840 s  (4.4 h)
  100,000,000 chunks ->    1,584,000 s  (440.0 h)


In [19]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))

# (a) position of the gold chunk before and after re-ranking
ids = [t["id"] for t in TEST_QUERIES]
before = [best_rank(results[i], t["gold_ids"], "bi_rank") or 11 for i, t in zip(ids, TEST_QUERIES)]
after = [best_rank(results[i], t["gold_ids"], "ce_rank") or 11 for i, t in zip(ids, TEST_QUERIES)]
x = np.arange(len(ids))
for xi, b0, a0 in zip(x, before, after):            # line from the baseline rank to the re-ranked rank
    ax[0].plot([xi, xi], [b0, a0], color="#555555", lw=1.5, zorder=1)
ax[0].scatter(x, before, s=70, color="#8c8c8c", label="baseline (Bi-Encoder)", zorder=2)
ax[0].scatter(x, after, s=70, color="#1f6fb4", marker="D", label="after Cross-Encoder", zorder=3)
ax[0].axhline(3.5, color="black", ls="--", lw=1)
ax[0].text(len(ids) - 0.6, 3.4, "top-3 cut-off (LLM context)", ha="right", va="bottom", fontsize=8)
ax[0].set_xticks(x); ax[0].set_xticklabels(ids)
ax[0].set_yticks(range(1, 12)); ax[0].set_yticklabels([*map(str, range(1, 11)), ">10"])
ax[0].set_ylabel("rank of gold chunk"); ax[0].legend(fontsize=8, loc="lower left"); ax[0].set_ylim(11.6, 0.4)
ax[0].set_title("Rank of the answer-bearing chunk")

# (b) cross-encoder cost grows linearly with the number of chunks it has to read
ax[1].plot(scale_df.iloc[:, 0], scale_df.iloc[:, 1], "o-", color="#1f6fb4", label="Cross-Encoder over n chunks")
ax[1].axhline(t_base.mean(), color="#8c8c8c", ls="--", label="Bi-Encoder + FAISS top-10")
ax[1].set_xlabel("chunks scored per query"); ax[1].set_ylabel("ms per query")
ax[1].set_title("Query-time cost"); ax[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "rank_shift_and_latency.png", dpi=150)
plt.show()

/var/folders/lp/whx9jwwx5kx8w81sn9w24d080000gn/T/ipykernel_29350/1505451020.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# Save everything needed for the report
export = []
for t in TEST_QUERIES:
    r = results[t["id"]]
    export.append({**{k: t[k] for k in ("id", "category", "query", "gold_phrase", "gold_ids")},
                   "top10": [{k: c[k] for k in ("chunk_id", "bi_rank", "bi_score", "ce_rank", "ce_score")}
                             for c in sorted(r["candidates"], key=lambda c: c["bi_rank"])],
                   "top3_baseline": r["top3_baseline"], "top3_rerank": r["top3_rerank"],
                   "answer_1": r["answer_1"], "answer_2": r["answer_2"], "llm_1": r["llm_1"], "llm_2": r["llm_2"]})
(RESULTS_DIR / "queries_and_answers.json").write_text(json.dumps(export, indent=2, ensure_ascii=False))
print("saved:", sorted(p.name for p in RESULTS_DIR.iterdir()))

saved: ['latency_end_to_end.csv', 'latency_retrieval.csv', 'llm_cache.json', 'queries_and_answers.json', 'rank_shift_and_latency.png', 'rank_shift_summary.csv', 'scaling_cross_encoder.csv']
